In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, os, json
 
from sklearn.model_selection   import train_test_split, StratifiedKFold
from sklearn.metrics           import (classification_report, roc_auc_score,
                                        confusion_matrix, precision_recall_curve,
                                        average_precision_score, RocCurveDisplay)
from sklearn.preprocessing     import label_binarize
from imblearn.over_sampling    import SMOTE
import xgboost as xgb
import shap
 
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)
 
OUTPUT_DIR = "Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ModuleNotFoundError: No module named 'imblearn'

In [3]:
print("\n" + "="*70)
print("STEP 1 — LOADING FEATURE TABLE")
print("="*70)
 
df = pd.read_csv(f"{OUTPUT_DIR}/model_ready_features.csv")
print(f"  Loaded: {df.shape[0]:,} members × {df.shape[1]} features")
 


STEP 1 — LOADING FEATURE TABLE


NameError: name 'OUTPUT_DIR' is not defined

In [3]:
print("\n" + "="*70)
print("STEP 2 — PREPARING TRAIN/TEST DATA")
print("="*70)
 
# Features to use in the model
DROP_COLS = [
    "loyalty_number",   # ID — not a feature
    "churn_label",      # target
    "never_flew_flag",  # perfectly correlated with Dormant — data leakage risk
]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
 
# Filter to Active + Churned only
model_df = df[df["churn_label"].isin([0, 1])].copy()
print(f"  Model dataset: {len(model_df):,} members (Active + Churned)")
print(f"  Class distribution:")
print(f"    Active  (0): {(model_df['churn_label']==0).sum():,}  "
      f"({(model_df['churn_label']==0).mean()*100:.1f}%)")
print(f"    Churned (1): {(model_df['churn_label']==1).sum():,}  "
      f"({(model_df['churn_label']==1).mean()*100:.1f}%)")
 
X = model_df[FEATURE_COLS]
y = model_df["churn_label"]
 
# Time-aware split: since we have limited temporal data,
# we use stratified random split (80/20) preserving class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\n  Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"  Train churn rate: {y_train.mean()*100:.1f}%")
print(f"  Test  churn rate: {y_test.mean()*100:.1f}%")
 


STEP 2 — PREPARING TRAIN/TEST DATA
  Model dataset: 15,167 members (Active + Churned)
  Class distribution:
    Active  (0): 14,268  (94.1%)
    Churned (1): 899  (5.9%)

  Train: 12,133 | Test: 3,034
  Train churn rate: 5.9%
  Test  churn rate: 5.9%


In [4]:
print("\n" + "="*70)
print("STEP 3 — SMOTE OVERSAMPLING (train set only)")
print("="*70)
 
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
 
print(f"  Before SMOTE — Active: {(y_train==0).sum():,} | "
      f"Churned: {(y_train==1).sum():,}")
print(f"  After  SMOTE — Active: {(y_train_bal==0).sum():,} | "
      f"Churned: {(y_train_bal==1).sum():,}")
 


STEP 3 — SMOTE OVERSAMPLING (train set only)
  Before SMOTE — Active: 11,414 | Churned: 719
  After  SMOTE — Active: 11,414 | Churned: 11,414


In [5]:
print("\n" + "="*70)
print("STEP 4 — TRAINING XGBOOST MODEL")
print("="*70)
 
# scale_pos_weight on original imbalance ratio as secondary guard
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count
 
model = xgb.XGBClassifier(
    n_estimators      = 300,
    max_depth         = 5,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    gamma             = 1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    scale_pos_weight  = scale_pos,   # extra guard on top of SMOTE
    eval_metric       = "auc",
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1
)
 
# Fit with early stopping on a validation slice
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_bal, y_train_bal, test_size=0.1, stratify=y_train_bal, random_state=42
)
 
model.fit(
    X_tr, y_tr,
    eval_set        = [(X_val, y_val)],
    verbose         = 50
)
 
print(f"\n  Model trained ✓  ({model.n_estimators} trees)")
 


STEP 4 — TRAINING XGBOOST MODEL
[0]	validation_0-auc:0.97760
[50]	validation_0-auc:0.99187
[100]	validation_0-auc:0.99412
[150]	validation_0-auc:0.99523
[200]	validation_0-auc:0.99581
[250]	validation_0-auc:0.99624
[299]	validation_0-auc:0.99654

  Model trained ✓  (300 trees)


In [6]:
print("\n" + "="*70)
print("STEP 5 — MODEL EVALUATION")
print("="*70)
 
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= 0.5).astype(int)
 
# ── Core metrics ─────────────────────────────────────────────────────────────
auc_roc = roc_auc_score(y_test, y_pred_proba)
auc_pr  = average_precision_score(y_test, y_pred_proba)
cm      = confusion_matrix(y_test, y_pred)
report  = classification_report(y_test, y_pred,
                                  target_names=["Active", "Churned"])
 
print(f"\n  AUC-ROC : {auc_roc:.4f}  (1.0 = perfect, 0.5 = random)")
print(f"  AUC-PR  : {auc_pr:.4f}  (better metric for imbalanced data)")
print(f"\n  Confusion Matrix:")
print(f"  {'':20s} Predicted Active  Predicted Churned")
print(f"  Actual Active  :  {cm[0,0]:>10,}        {cm[0,1]:>10,}")
print(f"  Actual Churned :  {cm[1,0]:>10,}        {cm[1,1]:>10,}")
print(f"\n  Classification Report:")
print(report)
 
# ── Lift at top 20% ──────────────────────────────────────────────────────────
# How many real churners are captured if we target top 20% by score?
n_top20    = int(len(y_test) * 0.20)
top20_idx  = y_pred_proba.argsort()[::-1][:n_top20]
lift_20    = y_test.iloc[top20_idx].mean() / y_test.mean()
churners_captured_20 = y_test.iloc[top20_idx].sum() / y_test.sum()
 
print(f"\n  BUSINESS METRICS:")
print(f"  Targeting top 20% by churn score:")
print(f"    Lift              : {lift_20:.1f}×  "
      f"(top 20% are {lift_20:.1f}× more likely to churn than average)")
print(f"    Churners captured : {churners_captured_20*100:.1f}% of all churners")
 
# ── Precision at different thresholds ────────────────────────────────────────
print(f"\n  Precision vs Recall at different score thresholds:")
print(f"  {'Threshold':>12} {'Precision':>12} {'Recall':>12} {'Flagged':>12}")
print(f"  {'-'*52}")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_t     = (y_pred_proba >= thresh).astype(int)
    flagged    = pred_t.sum()
    if flagged == 0:
        continue
    tp = ((pred_t == 1) & (y_test == 1)).sum()
    fp = ((pred_t == 1) & (y_test == 0)).sum()
    fn = ((pred_t == 0) & (y_test == 1)).sum()
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"  {thresh:>12.1f} {prec:>12.3f} {recall:>12.3f} {flagged:>12,}")
 


STEP 5 — MODEL EVALUATION

  AUC-ROC : 0.9208  (1.0 = perfect, 0.5 = random)
  AUC-PR  : 0.7749  (better metric for imbalanced data)

  Confusion Matrix:
                       Predicted Active  Predicted Churned
  Actual Active  :       2,666               188
  Actual Churned :          37               143

  Classification Report:
              precision    recall  f1-score   support

      Active       0.99      0.93      0.96      2854
     Churned       0.43      0.79      0.56       180

    accuracy                           0.93      3034
   macro avg       0.71      0.86      0.76      3034
weighted avg       0.95      0.93      0.94      3034


  BUSINESS METRICS:
  Targeting top 20% by churn score:
    Lift              : 4.2×  (top 20% are 4.2× more likely to churn than average)
    Churners captured : 84.4% of all churners

  Precision vs Recall at different score thresholds:
     Threshold    Precision       Recall      Flagged
  ---------------------------------------

In [7]:
print("\n" + "="*70)
print("STEP 6 — SHAP FEATURE IMPORTANCE")
print("="*70)
 
explainer  = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
 
# Mean absolute SHAP per feature
shap_importance = pd.DataFrame({
    "feature"          : FEATURE_COLS,
    "mean_abs_shap"    : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
 
print(f"\n  Top 15 features by SHAP importance:")
print(shap_importance.head(15).to_string(index=False))
 
# Save full SHAP importance
shap_importance.to_csv(f"{OUTPUT_DIR}/shap_feature_importance.csv", index=False)
 


STEP 6 — SHAP FEATURE IMPORTANCE

  Top 15 features by SHAP importance:
                 feature  mean_abs_shap
months_since_last_flight       2.280836
        active_months_3m       0.606131
enrollment_tenure_months       0.444054
  marital_status_Married       0.349881
       active_months_18m       0.316838
       education_encoded       0.298622
                     clv       0.259815
            flight_trend       0.246645
         pts_redeemed_3m       0.241710
             gender_Male       0.231114
   marital_status_Single       0.226953
             distance_6m       0.209981
            distance_12m       0.208517
        active_months_6m       0.198385
             flights_18m       0.194918


In [8]:
print("\n" + "="*70)
print("STEP 7 — SCORING ALL MEMBERS")
print("="*70)
 
# Score the full dataset
X_all   = df[FEATURE_COLS]
scores  = model.predict_proba(X_all)[:, 1]
 
predictions = df[["loyalty_number", "churn_label"]].copy()
predictions["churn_score"] = scores
 
# For Dormant members: override score to 1.0 (they are definitionally disengaged)
predictions.loc[df["churn_label"] == 2, "churn_score"] = 1.0
 
# Risk tier based on score
def assign_risk_tier(score, churn_label):
    if churn_label == 2:
        return "Dormant"
    elif score >= 0.70:
        return "High Risk"
    elif score >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"
 
predictions["risk_tier"] = predictions.apply(
    lambda r: assign_risk_tier(r["churn_score"], r["churn_label"]), axis=1
)
 
# Merge back with loyalty for context
loyalty_v2 = pd.read_csv(f"{OUTPUT_DIR}/loyalty_cleaned_v2.csv")
predictions = predictions.merge(
    loyalty_v2[["loyalty_number", "loyalty_card", "clv",
                "enrollment_year", "province"]],
    on="loyalty_number", how="left"
)
 
# Risk tier summary
print(f"\n  Risk Tier Distribution (all 16,737 members):")
tier_summary = predictions["risk_tier"].value_counts()
for tier, count in tier_summary.items():
    pct = count / len(predictions) * 100
    print(f"    {tier:<15}: {count:>6,}  ({pct:.1f}%)")
 
# Top 20 highest-risk active members (excluding Dormant)
top_at_risk = (predictions[predictions["churn_label"] != 2]
               .sort_values("churn_score", ascending=False)
               .head(20))
print(f"\n  Top 20 highest-risk active members:")
print(top_at_risk[["loyalty_number", "loyalty_card", "clv",
                    "churn_score", "risk_tier"]].to_string(index=False))
 


STEP 7 — SCORING ALL MEMBERS

  Risk Tier Distribution (all 16,737 members):
    Low Risk       : 13,039  (77.9%)
    Dormant        :  1,570  (9.4%)
    High Risk      :  1,203  (7.2%)
    Medium Risk    :    925  (5.5%)

  Top 20 highest-risk active members:
 loyalty_number loyalty_card      clv  churn_score risk_tier
         835295         Star 24127.50     0.999925 High Risk
         480021         Nova  7028.94     0.999919 High Risk
         746580         Nova  8237.04     0.999913 High Risk
         979330         Star  3677.91     0.999913 High Risk
         680304         Star  2706.61     0.999912 High Risk
         444661         Star  6701.77     0.999910 High Risk
         214123         Star 16229.65     0.999909 High Risk
         766691         Star  8281.74     0.999908 High Risk
         187161         Nova  3621.69     0.999906 High Risk
         110030         Nova  3387.78     0.999903 High Risk
         888901         Star  6819.23     0.999901 High Risk
      

In [9]:
print("\n" + "="*70)
print("STEP 8 — SAVING PLOTS")
print("="*70)
 
# Plot 1: ROC Curve
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=ax,
                                   color="steelblue", name=f"XGBoost (AUC={auc_roc:.3f})")
ax.plot([0,1],[0,1], "k--", linewidth=0.8, label="Random")
ax.set_title("ROC Curve — Churn Model", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_model_roc_curve.png", dpi=120)
plt.close()
print(f"  Saved: plot_model_roc_curve.png")
 
# Plot 2: Precision-Recall Curve
precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_pred_proba)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall_arr, precision_arr, color="coral", linewidth=2,
        label=f"XGBoost (AP={auc_pr:.3f})")
ax.axhline(y_test.mean(), color="grey", linestyle="--",
           label=f"Baseline ({y_test.mean():.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — Churn Model", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_model_pr_curve.png", dpi=120)
plt.close()
print(f"  Saved: plot_model_pr_curve.png")
 
# Plot 3: Top 15 SHAP Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
top15 = shap_importance.head(15)
bars  = ax.barh(top15["feature"][::-1], top15["mean_abs_shap"][::-1],
                color="steelblue", edgecolor="white")
ax.set_title("Top 15 Features by Mean |SHAP| Value", fontweight="bold")
ax.set_xlabel("Mean |SHAP| (impact on churn prediction)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_model_shap_importance.png", dpi=120)
plt.close()
print(f"  Saved: plot_model_shap_importance.png")
 
# Plot 4: Churn score distribution by true label
fig, ax = plt.subplots(figsize=(10, 5))
active_scores  = predictions[predictions["churn_label"] == 0]["churn_score"]
churned_scores = predictions[predictions["churn_label"] == 1]["churn_score"]
ax.hist(active_scores,  bins=40, alpha=0.5, color="steelblue",
        label="Active",  density=True, edgecolor="none")
ax.hist(churned_scores, bins=40, alpha=0.5, color="coral",
        label="Churned", density=True, edgecolor="none")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Threshold=0.5")
ax.set_title("Churn Score Distribution: Active vs Churned Members",
             fontweight="bold")
ax.set_xlabel("Churn Score (probability of churning)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_model_score_distribution.png", dpi=120)
plt.close()
print(f"  Saved: plot_model_score_distribution.png")
 
# Plot 5: Risk tier breakdown
fig, ax = plt.subplots(figsize=(8, 5))
tier_order  = ["Low Risk", "Medium Risk", "High Risk", "Dormant"]
tier_colors = ["steelblue", "gold", "coral", "grey"]
tier_counts = [tier_summary.get(t, 0) for t in tier_order]
bars = ax.bar(tier_order, tier_counts, color=tier_colors, edgecolor="white")
for bar, val in zip(bars, tier_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,}\n({val/len(predictions)*100:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_title("Member Risk Tier Distribution", fontweight="bold")
ax.set_ylabel("Members")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_model_risk_tiers.png", dpi=120)
plt.close()
print(f"  Saved: plot_model_risk_tiers.png")
 


STEP 8 — SAVING PLOTS
  Saved: plot_model_roc_curve.png
  Saved: plot_model_pr_curve.png
  Saved: plot_model_shap_importance.png
  Saved: plot_model_score_distribution.png
  Saved: plot_model_risk_tiers.png


In [10]:
print("\n" + "="*70)
print("STEP 9 — SAVING MODEL OUTPUTS")
print("="*70)
 
# Save predictions
predictions.to_csv(f"{OUTPUT_DIR}/churn_predictions.csv", index=False)
print(f"  Saved: churn_predictions.csv  ({len(predictions):,} members)")
 
# Save model
model.save_model(f"{OUTPUT_DIR}/churn_model.json")
print(f"  Saved: churn_model.json")
 
# Save model report
report_text = f"""
CHURN MODEL REPORT
==================
Model       : XGBoost Classifier
Date        : 2024
Dataset     : {len(model_df):,} members (Active + Churned)
Features    : {len(FEATURE_COLS)}
Train/Test  : 80% / 20% stratified split
Imbalance   : Handled with SMOTE + scale_pos_weight={scale_pos:.1f}
 
PERFORMANCE METRICS
-------------------
AUC-ROC             : {auc_roc:.4f}
AUC-PR              : {auc_pr:.4f}
Lift @ top 20%      : {lift_20:.1f}x
Churners captured   : {churners_captured_20*100:.1f}% in top 20% scored members
 
CLASSIFICATION REPORT (threshold = 0.5)
{report}
 
TOP 10 FEATURES (SHAP)
{shap_importance.head(10).to_string(index=False)}
 
RISK TIER SUMMARY (all {len(predictions):,} members)
{tier_summary.to_string()}
"""
 
with open(f"{OUTPUT_DIR}/model_report.txt", "w") as f:
    f.write(report_text)
print(f"  Saved: model_report.txt")
 
print(f"""
{"="*70}
PHASE 3 COMPLETE
{"="*70}
 
  KEY RESULTS:
  ─────────────────────────────────────────────────────
  AUC-ROC  : {auc_roc:.3f}   (target: > 0.75)
  AUC-PR   : {auc_pr:.3f}   (target: > 0.30)
  Lift @20%: {lift_20:.1f}×     (target: > 2×)
  ─────────────────────────────────────────────────────
 
  NEXT STEP → Phase 4: Customer Segmentation
  Run: python phase4_segmentation.py
""")
 


STEP 9 — SAVING MODEL OUTPUTS
  Saved: churn_predictions.csv  (16,737 members)
  Saved: churn_model.json
  Saved: model_report.txt

PHASE 3 COMPLETE

  KEY RESULTS:
  ─────────────────────────────────────────────────────
  AUC-ROC  : 0.921   (target: > 0.75)
  AUC-PR   : 0.775   (target: > 0.30)
  Lift @20%: 4.2×     (target: > 2×)
  ─────────────────────────────────────────────────────

  NEXT STEP → Phase 4: Customer Segmentation
  Run: python phase4_segmentation.py

